# EDA — Chicago Crimes Dataset
**CC65 Programación Concurrente y Distribuida**

Este notebook analiza los outputs generados por el loader en Go (`output/`) para:
1. Validar la calidad del proceso de limpieza
2. Identificar los features más relevantes para el modelo ML
3. Responder la pregunta de impacto social:

> *¿En qué horas y distritos hay mayor riesgo de crimen en Chicago para optimizar el patrullaje preventivo?*

**Archivos que usa este notebook** (todos dentro de `output/`, generados por Go):
- `load_stats.json` — métricas del proceso de carga concurrente
- `clean_records_sample.json` — 10,000 registros limpios (estructura `CleanRecord`)
- `risk_analysis.json` — agregados calculados sobre los 8.5M registros completos

## 0. Setup

In [ ]:
# Si no tienes las librerías, descomenta y ejecuta:
# !pip install pandas matplotlib seaborn

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Ruta al output del loader Go (relativa a la carpeta eda/)
OUTPUT_DIR = Path("../output")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("Librerías cargadas ✓")
print(f"Output dir: {OUTPUT_DIR.resolve()}")

## 1. Carga de datos

In [ ]:
# Métricas del loader Go
with open(OUTPUT_DIR / "load_stats.json") as f:
    stats = json.load(f)

# Sample de 10k registros limpios (CleanRecord)
with open(OUTPUT_DIR / "clean_records_sample.json") as f:
    sample_data = json.load(f)
df = pd.DataFrame(sample_data)

# Agregados sobre los 8.5M registros completos
with open(OUTPUT_DIR / "risk_analysis.json") as f:
    risk = json.load(f)

print("=" * 50)
print("MÉTRICAS DEL LOADER (Go — carga concurrente)")
print("=" * 50)
print(f"  Total registros procesados : {stats['total_lines']:,}")
print(f"  Registros válidos          : {stats['valid_records']:,}")
print(f"  Registros inválidos        : {stats['invalid_records']:,}")
print(f"  Tasa de éxito              : {stats['valid_records']/stats['total_lines']:.2%}")
print(f"  Workers goroutines usados  : {stats['workers_used']}")
print(f"  Tiempo de carga            : {stats['elapsed_ms']} ms")
print(f"  Velocidad                  : {stats['records_per_sec']:,.0f} registros/seg")
print()
print(f"Sample cargado: {len(df):,} registros")

## 2. Estructura del CleanRecord

In [ ]:
print("Tipos de datos:")
print(df.dtypes)
print()
df.head(5)

In [ ]:
# Verificar nulos — el cleaner.go debería haber eliminado todos
nulls = df.isnull().sum()
print("Valores nulos por columna:")
if nulls.sum() == 0:
    print("  Ninguno — limpieza en Go exitosa ✓")
else:
    print(nulls[nulls > 0])

In [ ]:
# Estadísticas descriptivas de variables numéricas
df[["hour", "day_of_week", "month", "year", "district", "community_area", "beat"]].describe().round(2)

## 3. Distribución temporal
Usamos los agregados de `risk_analysis.json` que cubren los **8.5M registros completos**.

In [ ]:
# Crímenes por hora
hours = {int(k): v for k, v in risk["crimes_by_hour"].items()}
hours_df = pd.DataFrame(sorted(hours.items()), columns=["hour", "count"])

peak_hour = hours_df.loc[hours_df["count"].idxmax(), "hour"]
min_hour  = hours_df.loc[hours_df["count"].idxmin(), "hour"]

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(hours_df["hour"], hours_df["count"] / 1e3,
       color=sns.color_palette("muted")[0], edgecolor="white")
ax.bar(peak_hour, hours_df.loc[hours_df["hour"]==peak_hour, "count"].values[0]/1e3,
       color="#e74c3c", label=f"Pico: {peak_hour}:00h")
ax.bar(min_hour, hours_df.loc[hours_df["hour"]==min_hour, "count"].values[0]/1e3,
       color="#2ecc71", label=f"Mínimo: {min_hour}:00h")
ax.set_xlabel("Hora del día")
ax.set_ylabel("Crímenes (miles)")
ax.set_title(f"Crímenes por hora del día — Chicago 2001-2026 ({risk['total_records']:,} registros)")
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

high_risk = hours_df[hours_df["count"] > hours_df["count"].quantile(0.75)]["hour"].tolist()
print(f"Hora pico   : {peak_hour}:00h ({hours_df[hours_df['hour']==peak_hour]['count'].values[0]:,} crímenes)")
print(f"Hora mínima : {min_hour}:00h ({hours_df[hours_df['hour']==min_hour]['count'].values[0]:,} crímenes)")
print(f"Franjas alto riesgo (top 25%): {sorted(high_risk)}")

In [ ]:
# Crímenes por día de semana
days_map = {0:"Dom", 1:"Lun", 2:"Mar", 3:"Mié", 4:"Jue", 5:"Vie", 6:"Sáb"}
dow = {int(k): v for k, v in risk["crimes_by_day_of_week"].items()}
dow_df = pd.DataFrame([(days_map[k], v) for k, v in sorted(dow.items())],
                      columns=["day", "count"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dow_df["day"], dow_df["count"] / 1e3,
       color=sns.color_palette("muted")[2], edgecolor="white")
ax.axhline(dow_df["count"].mean()/1e3, color="red", linestyle="--",
           alpha=0.6, label="Promedio")
ax.set_xlabel("Día de la semana")
ax.set_ylabel("Crímenes (miles)")
ax.set_title("Crímenes por día de la semana")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Distribución geográfica — Distritos

In [ ]:
risk_dist = pd.DataFrame(risk["top_risk_districts"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].barh(risk_dist["district"].astype(str)[::-1],
             risk_dist["crime_count"][::-1] / 1e3,
             color=sns.color_palette("muted")[3])
axes[0].set_xlabel("Crímenes (miles)")
axes[0].set_title("Volumen de crímenes por distrito")

axes[1].barh(risk_dist["district"].astype(str)[::-1],
             risk_dist["arrest_rate"][::-1] * 100,
             color=sns.color_palette("muted")[1])
axes[1].set_xlabel("Tasa de arresto (%)")
axes[1].set_title("Tasa de arresto por distrito")

plt.suptitle("Top distritos — Chicago", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Tipos de crimen

In [ ]:
types_df = pd.DataFrame(sorted(risk["crimes_by_type"].items(), key=lambda x: -x[1]),
                        columns=["type", "count"])
types_df["pct"] = (types_df["count"] / types_df["count"].sum() * 100).round(2)
top10 = types_df.head(10)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top10["type"][::-1], top10["count"][::-1] / 1e3,
        color=sns.color_palette("muted", 10)[::-1])
ax.set_xlabel("Crímenes (miles)")
ax.set_title("Top 10 tipos de crimen — Chicago 2001-2026")
plt.tight_layout()
plt.show()

print(f"Cobertura del top 10: {top10['pct'].sum():.1f}% del total")
print()
print(top10[["type", "count", "pct"]].to_string(index=False))

## 6. Heatmap hora × distrito
Muestra exactamente en qué combinación hora+distrito se concentra el crimen — responde directamente la pregunta de impacto social.

In [ ]:
heatmap_df = pd.DataFrame(risk["heatmap_data"])
pivot = heatmap_df.pivot_table(index="district", columns="hour",
                                values="count", aggfunc="sum").fillna(0)

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(pivot / 1e3, cmap="YlOrRd", ax=ax,
            linewidths=0.3, linecolor="white",
            cbar_kws={"label": "Crímenes (miles)"})
ax.set_xlabel("Hora del día")
ax.set_ylabel("Distrito")
ax.set_title("Densidad de crímenes: Hora × Distrito — Chicago 2001-2026")
plt.tight_layout()
plt.show()

max_idx = heatmap_df.loc[heatmap_df["count"].idxmax()]
print(f"Combinación de máximo riesgo: Distrito {int(max_idx['district'])}, "
      f"Hora {int(max_idx['hour'])}:00h ({int(max_idx['count']):,} crímenes)")

## 7. Features para ML — análisis del sample

In [ ]:
arrest_rate = df["arrest"].mean()
print(f"Tasa de arresto global : {arrest_rate:.2%}")
print(f"Crímenes domésticos    : {df['domestic'].mean():.2%}")
print()
print("Cardinalidad de variables categóricas:")
for col in ["primary_type", "location_description", "district", "beat", "community_area"]:
    print(f"  {col:<25}: {df[col].nunique()} valores únicos")

In [ ]:
# Tasa de arresto por tipo de crimen
arrest_by_type = (df.groupby("primary_type")["arrest"]
                    .agg(["mean", "count"])
                    .rename(columns={"mean": "arrest_rate", "count": "n"})
                    .query("n >= 20")
                    .sort_values("arrest_rate", ascending=False))

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(arrest_by_type.head(12).index[::-1],
        arrest_by_type.head(12)["arrest_rate"][::-1] * 100,
        color=sns.color_palette("muted")[1])
ax.axvline(arrest_rate * 100, color="red", linestyle="--",
           alpha=0.7, label=f"Promedio global: {arrest_rate:.1%}")
ax.set_xlabel("Tasa de arresto (%)")
ax.set_title("Tasa de arresto por tipo de crimen (variable objetivo del modelo ML)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlación entre variables numéricas
num_cols = ["hour", "day_of_week", "month", "year",
            "district", "community_area", "beat", "arrest", "domestic"]
df_num = df[num_cols].copy()
df_num["arrest"]   = df_num["arrest"].astype(int)
df_num["domestic"] = df_num["domestic"].astype(int)

corr = df_num.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title("Correlación entre variables numéricas")
plt.tight_layout()
plt.show()

print("Correlación con 'arrest' (variable objetivo):")
arrest_corr = corr["arrest"].drop("arrest").abs().sort_values(ascending=False)
for feat, val in arrest_corr.items():
    print(f"  {feat:<20}: {val:.4f}")

## 8. Resumen — Features recomendados para el modelo ML

In [ ]:
print("""
╔══════════════════════════════════════════════════════════╗
║         FEATURES RECOMENDADOS PARA EL MODELO ML          ║
╠══════════════════════════════════════════════════════════╣
║  Variable objetivo (target):                             ║
║    arrest  →  clasificación binaria 0/1                  ║
║                                                          ║
║  Features de entrada:                                    ║
║    Temporales  : hour, day_of_week, month                ║
║    Geográficos : district, community_area, beat          ║
║    Categóricos : primary_type, location_description      ║
║    Binarios    : domestic                                ║
║                                                          ║
║  Descartar: id, year, latitude, longitude                ║
║    (no predictivos o reemplazados por geográficos)       ║
╠══════════════════════════════════════════════════════════╣
║  Pregunta de impacto social:                             ║
║  ¿En qué horas y distritos hay mayor riesgo de crimen   ║
║  en Chicago para optimizar el patrullaje preventivo?    ║
╚══════════════════════════════════════════════════════════╝
""")